# 02 - RoPE (rotary position) from scratch

**What:** RoPE encodes WHERE a token is by ROTATING pairs of its q/k dimensions. Attention then sees relative position as a pure function of the angle difference.

**Why:** it replaces the learned position table of GPT-2 (experiments B and D), and its defining property - `<R(m)q, R(n)k> = <R(m-n)q, k>` - is verified below, not assumed.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))   # repo root, so `src` imports work
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # Windows OMP clash
print("repo on path:", os.path.abspath('..'))

In [ ]:
import torch, math

# --- 1. THE FREQUENCIES: base^(-2i/d) for each dimension PAIR -------------
d = 16                      # head dim -> 8 pairs
base = 10000.0
i = torch.arange(0, d // 2).float()
inv_freq = 1.0 / (base ** (i / (d // 2)))
print("rotation frequencies per pair:", inv_freq.tolist())
print("pair 0 rotates slowly (long-range), pair 7 fast (local)")

In [ ]:
# --- 2. THE TABLES: angle(position, pair) -> cos/sin ----------------------
def build_tables(max_len, d, base=10000.0):
    i = torch.arange(0, d // 2).float()
    inv_freq = 1.0 / (base ** (i / (d // 2)))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)  # (T, d/2)
    emb = torch.cat([angles, angles], dim=-1)                      # (T, d)
    return emb.cos(), emb.sin()

cos, sin = build_tables(16, d)
print("cos table:", tuple(cos.shape), " (T, d)")

def rotate_half(x):                      # (x1, x2) -> (-x2, x1)
    x1, x2 = x[..., :d // 2], x[..., d // 2:]
    return torch.cat([-x2, x1], dim=-1)

def rotate(x, pos):                      # 2D rotation by the row `pos`
    return x * cos[pos] + rotate_half(x) * sin[pos]

In [ ]:
# --- 3. WATCH one vector rotate as position grows -------------------------
torch.manual_seed(0)
x = torch.randn(2, d)                    # first two dims = one rotation pair
for pos in range(6):
    r = rotate(x, pos)
    print(f"position {pos}:  dims 0-1 = ({r[0,0]:+.3f}, {r[0,1]:+.3f})  "
          f"| length {r.norm():.4f}")
print("(length never changes - rotation is shape-preserving, "
      "so it cannot distort attention scores)")

In [ ]:
# --- 4. THE DEFINING PROPERTY: relative position, not absolute -------------
torch.manual_seed(3)
q0 = torch.randn(1, d)
k0 = torch.randn(1, d)

def score(m, n):                          # attention score at positions m, n
    return (rotate(q0, m) * rotate(k0, n)).sum()

print("scores for the same DISTANCE apart (must be equal):")
for dist in range(4):
    print(f"  distance {dist}: (0,{dist})={score(0, dist):+.3f}   "
          f"({3},{3+dist})={score(3, 3 + dist):+.3f}")
print()
print("only the gap matters, not where it is in the sentence: "
      "that is RoPE.")

In [ ]:
# --- 5. PARAMETER COUNT: learned table vs rotation ------------------------
from src.model.gpt import GPT, GPTConfig
m_learned = GPT(GPTConfig(vocab_size=1000, block_size=256, n_layer=2,
                          n_head=2, n_embd=32, position="learned"))
m_rope = GPT(GPTConfig(vocab_size=1000, block_size=256, n_layer=2,
                       n_head=2, n_embd=32, position="rope"))
diff = m_learned.get_num_params(False) - m_rope.get_num_params(False)
print(f"learned-position params: {m_learned.get_num_params(False):,}")
print(f"rope params            : {m_rope.get_num_params(False):,}")
print(f"difference             : {diff:,} = block_size x n_embd "
      f"(the position table that RoPE deletes)")

**Takeaway:** RoPE gets position for FREE (no table, no extra params) and generalizes to lengths beyond training because a rotation angle exists for every position. Whether that translates into better loss at equal tokens is exactly what experiments B/D answer.